**Part 2 · LangChain** · step 14 of 25  
[← 13 Middleware: summarization & PII](13_middleware_summarization_and_pii.ipynb) · [🏠 Roadmap](../README.md) · [15 Custom middleware →](15_custom_middleware.ipynb)

# 14 · Human-in-the-Loop (HITL)

Some tool calls are risky (sending emails, payments, deleting data). `HumanInTheLoopMiddleware`
**pauses** the agent *before* those tools run and waits for a human decision.

```
agent.invoke(request) ──► model wants send_email ──► ⏸ INTERRUPT (result["__interrupt__"])
agent.invoke(Command(resume={"decisions": [...]})) ──► approve / edit / reject ──► continue
```

Requirements:
1. A **checkpointer** — the paused state must be saved somewhere.
2. The **same `thread_id`** when resuming.

| Decision | Effect |
|---|---|
| `{"type": "approve"}` | run the tool with the model's arguments |
| `{"type": "edit", "edited_action": {"name": ..., "args": {...}}}` | run the tool with **your** arguments |
| `{"type": "reject", "message": "why"}` | don't run it; the reason is sent back to the model |

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def read_email(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    print(f"Sending email to {recipient} | subject={subject!r} | body={body!r}")
    return f"Email sent to {recipient} with subject {subject}"


def build_agent():
    """Create an email agent that needs approval before ``send_email`` runs."""
    return create_agent(
        model="gpt-4o",
        tools=[read_email, send_email],
        checkpointer=InMemorySaver(),
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},
                    "read_email": False,  # safe tool -> never interrupt
                }
            )
        ],
    )

## 1. Approve

In [ ]:
agent = build_agent()
config = {"configurable": {"thread_id": "hitl-approve"}}

result = agent.invoke(
    {"messages": [HumanMessage("Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config,
)

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
    print(f"✅ Result: {result['messages'][-1].content}")

### Inspecting a pause
While paused, `agent.get_state(config).interrupts` shows what is waiting for approval
(tool name, arguments and the allowed decisions).

In [ ]:
config = {"configurable": {"thread_id": "hitl-inspect"}}
agent.invoke({"messages": [HumanMessage("send email to test@mail.com with subject hello and body how are you")]}, config=config)

state = agent.get_state(config)
print(state.interrupts)

## 2. Edit — fix the arguments before the tool runs

In [ ]:
agent = build_agent()
config = {"configurable": {"thread_id": "hitl-edit"}}

# The user gave the wrong address...
result = agent.invoke(
    {"messages": [HumanMessage("Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config,
)

# ...so the human edits the action before approving it
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    result = agent.invoke(
        Command(resume={"decisions": [{
            "type": "edit",
            "edited_action": {
                "name": "send_email",
                "args": {
                    "recipient": "correct@email.com",
                    "subject": "Corrected Subject",
                    "body": "This was edited by human before sending",
                },
            },
        }]}),
        config=config,
    )
    print(f"✏️ Result: {result['messages'][-1].content}")

## 3. Reject — refuse and tell the model why

In [ ]:
agent = build_agent()
config = {"configurable": {"thread_id": "hitl-reject"}}

result = agent.invoke(
    {"messages": [HumanMessage("Send email to boss@company.com with subject 'I quit' and body 'Bye!'")]},
    config=config,
)

if "__interrupt__" in result:
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject", "message": "Do not send this, I was joking."}]}),
        config=config,
    )
    print(f"🚫 Result: {result['messages'][-1].content}")

---
✅ **Done with 14 Human-in-the-loop.** Next: [15 Custom middleware →](15_custom_middleware.ipynb)  
[🏠 Roadmap](../README.md)